# 02. Моделирование и MLflow

В этом ноутбуке строится baseline, обучается модель ранжирования клиент-продукт и рассчитываются ranking-метрики.

In [1]:
from pathlib import Path
import sys
import json

import pandas as pd

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.append(str(PROJECT_ROOT / "src"))

from bank_recs.config import PRODUCT_COLS, TOP_K
from bank_recs.data import clean_dataframe, load_raw_data, month_slice, available_months
from bank_recs.features import build_train_validation_sets, get_actual_new_products
from bank_recs.metrics import ranking_metrics
from bank_recs.modeling import fit_model, popularity_recommendations, predict_ranked_products, save_model

pd.set_option("display.max_columns", 100)

## Загрузка данных

In [2]:
data_path = PROJECT_ROOT / "data" / "raw" / "train_ver2.csv"
sample_path = PROJECT_ROOT / "data" / "raw" / "train_ver2_sample.csv"

if not data_path.exists():
    if not sample_path.exists():
        !python {PROJECT_ROOT / "scripts" / "make_sample_data.py"}
    data_path = sample_path

df = clean_dataframe(load_raw_data(data_path))
print(f"Используется файл: {data_path}")
print(df.shape)
print(available_months(df))

Используется файл: /Users/aleksandrfedyuk/Desktop/МАГА/bank-product-recommendation/data/raw/train_ver2.csv
(13647309, 51)
['2015-01', '2015-02', '2015-03', '2015-04', '2015-05', '2015-06', '2015-07', '2015-08', '2015-09', '2015-10', '2015-11', '2015-12', '2016-01', '2016-02', '2016-03', '2016-04', '2016-05']


## Временное разбиение

Для реального датасета удобно использовать:

- train: `2016-03 -> 2016-04`
- valid: `2016-04 -> 2016-05`

Если используется синтетический demo-файл, месяцы такие же.

In [3]:
TRAIN_HISTORY_MONTH = "2016-03"
TRAIN_TARGET_MONTH = "2016-04"
VALID_HISTORY_MONTH = "2016-04"
VALID_TARGET_MONTH = "2016-05"
MAX_USERS = 50_000  # для полного обучения можно поставить None в коде или 0 в train.py
NEGATIVE_RATIO = 5

## Формирование train/validation выборок

In [4]:
X_train, y_train, train_meta, X_valid, y_valid, valid_meta = build_train_validation_sets(
    df,
    TRAIN_HISTORY_MONTH,
    TRAIN_TARGET_MONTH,
    VALID_HISTORY_MONTH,
    VALID_TARGET_MONTH,
    negative_ratio=NEGATIVE_RATIO,
    max_users=MAX_USERS,
    random_state=42,
)

print("X_train:", X_train.shape)
print("Positive rate:", y_train.mean())
print("X_valid:", X_valid.shape)
X_train.head()

X_train: (107855, 24)
Positive rate: 0.016633443048537387
X_valid: (1133691, 24)


,age,antiguedad,renta,ind_nuevo,indrel,ind_actividad_cliente,months_from_join,prev_product_count,current_product_owned,product_popularity,ind_empleado,pais_residencia,sexo,indrel_1mes,tiprel_1mes,indresi,indext,conyuemp,canal_entrada,indfall,cod_prov,nomprov,segmento,product
0,36.0,6.0,38673.30,0.0,1.0,1.0,6.0,2,0,0.00006,N,ES,V,1,A,S,N,unknown,KHM,N,23.0,JAEN,02 - PARTICULARES,ind_aval_fin_ult1
1,36.0,6.0,38673.30,0.0,1.0,1.0,6.0,2,0,0.03868,N,ES,V,1,A,S,N,unknown,KHM,N,23.0,JAEN,02 - PARTICULARES,ind_tjcr_fin_ult1
2,46.0,6.0,31748.25,0.0,1.0,0.0,6.0,1,0,0.00130,N,ES,V,1,I,S,N,unknown,KHM,N,38.0,SANTA CRUZ DE TENERIFE,02 - PARTICULARES,ind_deme_fin_ult1
3,46.0,6.0,31748.25,0.0,1.0,0.0,6.0,1,0,0.12046,N,ES,V,1,I,S,N,unknown,KHM,N,38.0,SANTA CRUZ DE TENERIFE,02 - PARTICULARES,ind_recibo_ult1
4,29.0,6.0,83422.80,0.0,1.0,1.0,6.0,1,0,0.04800,N,ES,V,1,A,S,N,unknown,KHM,N,6.0,BADAJOZ,03 - UNIVERSITARIO,ind_reca_fin_ult1


## Baseline по популярности

In [5]:
valid_history = month_slice(df, VALID_HISTORY_MONTH).drop_duplicates("ncodpers")
valid_target = month_slice(df, VALID_TARGET_MONTH).drop_duplicates("ncodpers")

actual = get_actual_new_products(valid_history, valid_target)
baseline_pred = popularity_recommendations(valid_history, k=TOP_K)
baseline_metrics = ranking_metrics(actual, baseline_pred, PRODUCT_COLS, k=TOP_K)
baseline_metrics

{'map_at_7': 0.017703456440535743,
 'precision_at_7': 0.1758052530429523,
 'recall_at_7': 0.9500759342301945,
 'hitrate_at_7': 0.9543318385650225,
 'coverage_at_7': 0.875}

## Обучение модели

Модель обучается на строках формата `клиент-продукт`. Бинарный target показывает, появился ли продукт у клиента в следующем месяце.

In [6]:
model = fit_model(X_train, y_train, random_state=42)
model

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  ['age', 'antiguedad', 'renta',
                                                   'ind_nuevo', 'indrel',
                                                   'ind_actividad_cliente',
                                                   'months_from_join',
                                                   'prev_product_count',
                                                   'current_product_owned',
                                                   'product_popularity']),
                                                 ('cat',
                                                  Pipeline(steps=[('imp...
                                                                   FunctionTransformer(func=<function to_string_array at 0x12f7736a0>)),
                                                                  ('onehot',
                                                                   OneHotEncoder(handle_unknown='ignore',
                                                                                 min_frequency=10))]),
                                                  ['ind_empleado',
                                                   'pais_residencia', 'sexo',
                                                   'indrel_1mes', 'tiprel_1mes',
                                                   'indresi', 'indext',
                                                   'conyuemp', 'canal_entrada',
                                                   'indfall', 'cod_prov',
                                                   'nomprov', 'segmento',
                                                   'product'])])),
                ('classifier',
                 LogisticRegression(class_weight='balanced', max_iter=500,
                                    n_jobs=-1, random_state=42))])

## Оценка модели

In [7]:
model_pred = predict_ranked_products(model, X_valid, valid_meta, k=TOP_K)
model_metrics = ranking_metrics(actual, model_pred, PRODUCT_COLS, k=TOP_K)

comparison = pd.DataFrame([baseline_metrics, model_metrics], index=["popularity_baseline", "sklearn_model"])
comparison

,map_at_7,precision_at_7,recall_at_7,hitrate_at_7,coverage_at_7
popularity_baseline,0.017703,0.175805,0.950076,0.954332,0.875000
sklearn_model,0.001073,0.009737,0.053100,0.053238,0.791667


## Сохранение модели

In [8]:
model_path = PROJECT_ROOT / "models" / "model.joblib"
save_model(
    model,
    model_path,
    metadata={
        "top_k": TOP_K,
        "train_history_month": TRAIN_HISTORY_MONTH,
        "train_target_month": TRAIN_TARGET_MONTH,
        "valid_history_month": VALID_HISTORY_MONTH,
        "valid_target_month": VALID_TARGET_MONTH,
        "metrics": model_metrics,
    },
)
print(f"Модель сохранена: {model_path}")

Модель сохранена: /Users/aleksandrfedyuk/Desktop/МАГА/bank-product-recommendation/models/model.joblib


## MLflow

Основной способ запустить обучение с логированием в MLflow — использовать скрипт `scripts/train.py`.

1. В отдельном терминале запусти MLflow:

```bash
./scripts/run_mlflow.sh
```

2. Затем запусти обучение:

```bash
PYTHONPATH=src python scripts/train.py --data-path data/raw/train_ver2.csv
```

Или для demo-датасета:

```bash
python scripts/make_sample_data.py
PYTHONPATH=src python scripts/train.py --data-path data/raw/train_ver2_sample.csv --max-users 0
```

In [9]:
# Можно запустить обучение прямо из ноутбука на demo-датасете:
# !PYTHONPATH={PROJECT_ROOT / "src"} python {PROJECT_ROOT / "scripts" / "train.py"} --data-path {data_path} --max-users 0

## Выводы по моделированию

В этом ноутбуке задача была сформулирована как рекомендательная задача ранжирования новых банковских продуктов. Таргет строился как появление продукта у клиента в следующем месяце: если продукт отсутствовал в месяце `t`, но появился в месяце `t+1`, для пары `клиент × продукт` формировался положительный пример.

Для корректной проверки качества использовалось временное разбиение: признаки брались из одного месяца, а целевые события — из следующего. Такой подход снижает риск утечки информации из будущего и лучше соответствует production-сценарию, где модель должна рекомендовать продукты на будущий период.

Обучающая выборка была сформирована в формате `клиент × продукт`. В текущем эксперименте размер train-сета составил `107855` строк и `24` признака, validation-сета — `1133691` строк и `24` признака. Доля положительных примеров в train составила около `1.66%`, что подтверждает сильный дисбаланс классов: новые банковские продукты покупает небольшая доля клиентов.

В качестве baseline использовалась рекомендация наиболее популярных новых продуктов. Baseline показал:

* `MAP@7 = 0.0177`;
* `Precision@7 = 0.1758`;
* `Recall@7 = 0.9501`;
* `HitRate@7 = 0.9543`;
* `Coverage@7 = 0.8750`.

ML-модель показала:

* `MAP@7 = 0.0011`;
* `Precision@7 = 0.0097`;
* `Recall@7 = 0.0531`;
* `HitRate@7 = 0.0532`;
* `Coverage@7 = 0.7917`.

В текущем эксперименте популярностный baseline оказался сильнее ML-модели по основным ranking-метрикам. Это важный результат: простая стратегия рекомендаций часто является сильной точкой сравнения в рекомендательных системах, особенно при сильном дисбалансе классов и ограниченном наборе признаков. Полученный результат показывает, что перед усложнением модели необходимо улучшать feature engineering и схему обучения.

Возможные направления улучшения:

1. добавить лаговые признаки по продуктам за несколько предыдущих месяцев;
2. добавить агрегаты популярности продуктов по сегментам клиентов, возрастным группам, активности и провинциям;
3. увеличить обучающую выборку после проверки пайплайна на ограниченном числе пользователей;
4. настроить балансировку положительных и отрицательных примеров;
5. попробовать CatBoost/LightGBM вместо базовой sklearn-модели;
6. отдельно проанализировать ошибки модели по продуктам и клиентским сегментам.

Несмотря на то что ML-модель в текущем запуске уступает baseline, в проекте реализован полный воспроизводимый контур:

* формирование таргета как новых продуктов;
* временное train/validation-разбиение;
* baseline по популярности;
* ML-модель для ранжирования продуктов;
* ranking-метрики `MAP@7`, `Precision@7`, `Recall@7`, `HitRate@7`, `Coverage@7`;
* сохранение модели как артефакта;
* возможность логирования экспериментов через MLflow;
* дальнейшее использование модели в FastAPI-сервисе.
